# Clinical, Hormonal, and Genetic Characterization of Female Patients with Non-Classical 21-Hydroxylase Deficiency and Infertility Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is published as a Croissant schema and is accessible at the following URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```
It contains structured clinical, hormonal, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency and infertility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print summary information
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
We review available record sets, fields, and their `@id`s as defined by the Croissant schema.

> All entities are referenced by their `@id`.

Let's list all record sets (`@id`), their fields, and the field `@id`s.

In [ ]:
from mlcroissant._src.datasets.record_sets import RecordSet

# Get all record sets
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  RecordSet @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields (@id): {field_ids}\n")

## 3. Data Extraction
We load the data from each record set, referencing them by their exact `@id` for reproducibility.

Below we demonstrate how to extract all available record sets and convert them into pandas DataFrames for analysis.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record is a dictionary with keys as field @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")
    if df.shape[0] > 0:
        print(f"Columns: {df.columns.tolist()}")
    print()

## 4. Exploratory Data Analysis (EDA)
We select one of the clinical tabulation record sets for demonstration. For this
example, let's use the record set containing general clinical features. We'll identify a numeric field (e.g., age at diagnosis) via its field `@id`, then filter, normalize, and group the data for demonstration.

Remember: **All fields are referenced by their Croissant `@id`.**

In [ ]:
# Pick first record set with a numeric field (e.g., age)

# Find a clinical record set and its numeric field
clin_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in dataset.metadata.record_sets:
    # Search for field with data_type 'Integer' or 'Float'
    for field in rs.fields:
        # Try to find a plausible age field
        if 'age' in field.name.lower() or field.data_type in ('Integer', 'Float'):
            clin_record_set_id = rs.id
            numeric_field_id = field.id
            break
    if clin_record_set_id:
        # Attempt to group by a categorical (e.g. 'clitomegaly', 'hirsutism')
        for field in rs.fields:
            if field.data_type == 'Boolean' and group_field_id is None:
                group_field_id = field.id
        break

print(f"Selected RecordSet @id: {clin_record_set_id}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id (Boolean): {group_field_id}")

df = dataframes[clin_record_set_id]

# Demonstrate filtering and normalization
if numeric_field_id in df.columns:
    threshold = 20
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered rows where {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]])
    # Normalization (z-score)
    normalized_col = numeric_field_id + "_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' values:")
    print(filtered_df[[numeric_field_id, normalized_col]])
    # Group by Boolean field (if present)
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped by '{group_field_id}':")
        print(grouped)
else:
    print(f"No numeric field '{numeric_field_id}' in DataFrame.")

## 5. Visualization
We now visualize the numeric field distribution (e.g., age at diagnosis) and, if possible, compare distributions by group (e.g., presence/absence of a binary clinical feature).

This demonstration uses matplotlib/seaborn for visualization. If you filter on a different field, adjust the code as needed. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(5,3))
    sns.histplot(df[numeric_field_id], bins='auto', kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(5,3))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've:
- Loaded and explored a FAIR² dataset using the Croissant schema, referenced by `@id`s for each record set and field.
- Inspected the record set structure and extracted data as pandas DataFrames for analysis.
- Demonstrated basic exploratory data analysis: filtering and normalizing a numeric field and grouping by a Boolean field.
- Visualized a numeric variable, grouped by a clinical feature.

Due to the small cohort (N=3), this dataset is best suited for demonstration and clinical illustration rather than modeling. The mlcroissant library enables robust, reproducible access to structured biomedical data.